# 02 — Defender Model

This notebook walks through the four modules that make up the simulated defender:

| Module | Contents |
|---|---|
| `defender/honeypot.py` | Actions, threat-level computation, reward function |
| `defender/classifier.py` | RandomForest attack classifier |
| `defender/dqn.py` | DQN network, replay buffer, agent |
| `defender/defender.py` | Top-level orchestrator (classifier + DQN) |

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
sns.set_theme(style='whitegrid')

---
## 1. Honeypot Actions & Reward Function (`honeypot.py`)

The defender chooses one of five **honeypot actions** each step.  
The reward signal is shaped by the threat band (benign / low / medium / high / critical).

In [ ]:
from defender.honeypot import (
    HoneypotAction,
    threat_band,
    compute_threat_level,
    compute_reward,
    _REWARD_MATRIX,
)
from attacker.attack_types import AttackType, KillChainStage

print('Actions:', HoneypotAction.names())

### 1.1 Reward matrix heatmap

Rows = actions, columns = threat bands.

In [ ]:
bands = ['benign', 'low', 'medium', 'high', 'critical']
reward_data = np.array([
    [_REWARD_MATRIX[HoneypotAction(a)][b] for b in bands]
    for a in range(HoneypotAction.count())
])

fig, ax = plt.subplots(figsize=(8, 4))
sns.heatmap(
    reward_data, annot=True, fmt='.1f', cmap='RdYlGn', center=0,
    xticklabels=bands, yticklabels=HoneypotAction.names(), ax=ax
)
ax.set_title('Base Reward Matrix (action × threat band)', fontsize=12)
ax.set_xlabel('Threat band')
ax.set_ylabel('Honeypot action')
plt.tight_layout()
plt.show()

### 1.2 Threat level formula

```
threat = 0.45 × attack_severity
       + 0.35 × kill_chain_weight
       + 0.15 × escalation_rate
       + 0.05 × min(1, attack_count / 100)
```

In [ ]:
# Show how threat level varies across attack types at ACTIONS_ON_OBJ stage
stage = KillChainStage.ACTIONS_ON_OBJ
threat_levels = [
    compute_threat_level(at, stage, escalation_rate=0.5, attack_count=30)
    for at in AttackType
]

fig, ax = plt.subplots(figsize=(10, 4))
colors = ['green' if t < 0.35 else 'orange' if t < 0.55 else 'red' for t in threat_levels]
ax.bar(AttackType.names(), threat_levels, color=colors)
for th, label, ls in [(0.75, 'critical', 'r'), (0.55, 'high', 'darkorange'),
                       (0.35, 'medium', 'gold'), (0.15, 'low', 'steelblue')]:
    ax.axhline(th, linestyle='--', color=ls, alpha=0.7, label=label)
ax.set_title('Threat level at ACTIONS_ON_OBJ (escalation=0.5, count=30)')
ax.set_ylabel('Threat level')
ax.set_ylim(0, 1)
ax.tick_params(axis='x', rotation=45)
ax.legend(loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()

---
## 2. Attack Classifier (`classifier.py`)

A **RandomForest** trained on synthetic data from the Attacker's feature simulator.  
Class imbalance is handled via `class_weight='balanced'`.

In [ ]:
from defender.classifier import AttackClassifier

clf = AttackClassifier(n_estimators=100, max_depth=15)
print('Training classifier on 400 samples/class …')
clf.fit_from_simulation(n_samples_per_class=400, seed=42)
print('Done.')

In [ ]:
result = clf.evaluate(n_test_per_class=150, seed=99)
print(f"Test accuracy: {result['accuracy']:.3f}")

# Per-class precision/recall
report_df = pd.DataFrame(result['report']).T.drop(columns=['support'], errors='ignore')
report_df = report_df[report_df.index.isin(AttackType.names())]
report_df.plot(kind='bar', figsize=(12, 4))
plt.title('Classifier Per-class Precision / Recall / F1')
plt.ylabel('Score')
plt.ylim(0, 1.05)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Feature importances
from attacker.attack_types import FEATURE_NAMES

importances = clf.model.feature_importances_
imp_df = pd.Series(importances, index=FEATURE_NAMES).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
imp_df.plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('RandomForest Feature Importances')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

---
## 3. DQN Agent (`dqn.py`)

Key design choices:
- **Architecture**: fully connected with `LayerNorm` at each hidden layer  
- **Exploration**: epsilon-greedy with exponential decay  
- **Replay buffer**: fixed-capacity circular deque  
- **Loss**: Huber (SmoothL1) for robustness to outlier rewards  
- **Target network**: hard copy every `target_update_freq` steps

In [ ]:
from defender.dqn import DQNNetwork, DQNAgent
import torch

net = DQNNetwork(state_dim=24, action_dim=5, hidden_dims=[256, 128, 64])
print(net)

total_params = sum(p.numel() for p in net.parameters())
print(f'\nTotal parameters: {total_params:,}')

In [ ]:
# Instantiate agent and inspect epsilon schedule
agent = DQNAgent(
    state_dim=24, action_dim=5,
    epsilon_start=1.0, epsilon_end=0.05, epsilon_decay=0.997,
)

epsilons = []
eps = 1.0
for _ in range(2000):
    eps = max(0.05, eps * 0.997)
    epsilons.append(eps)

plt.figure(figsize=(9, 3))
plt.plot(epsilons, color='purple')
plt.axhline(0.05, linestyle='--', color='gray', label='ε_min = 0.05')
plt.title('Epsilon Decay Schedule (decay=0.997)')
plt.xlabel('Update step')
plt.ylabel('Epsilon')
plt.legend()
plt.tight_layout()
plt.show()

---
## 4. Defender Orchestrator (`defender.py`)

`Defender` wraps `AttackClassifier` + `DQNAgent` into a single object with
a clean `observe / learn / save / load` API.

In [ ]:
from defender.defender import Defender

defender = Defender(
    dqn_config={
        'state_dim': 24, 'action_dim': 5,
        'lr': 1e-3, 'gamma': 0.99,
        'epsilon_start': 1.0, 'epsilon_end': 0.05, 'epsilon_decay': 0.997,
        'batch_size': 64, 'target_update_freq': 150, 'buffer_capacity': 15_000,
    },
    classifier_config={'n_estimators': 100, 'max_depth': 15},
    train_classifier=True,
    seed=42,
)
defender.initialize_classifier(n_samples_per_class=300)

In [ ]:
# Single inference pass
dummy_state = np.zeros(24, dtype=np.float32)
dummy_state[0] = 1.0  # NORMAL attack one-hot
dummy_state[10] = 1.0  # RECONNAISSANCE stage

dummy_features = {f: 0.0 for f in FEATURE_NAMES}
dummy_features['dur'] = 2.5
dummy_features['sload'] = 3000.0

action, predicted_attack = defender.observe(dummy_state, dummy_features, training=False)
print(f'Action chosen    : {HoneypotAction(action).name}')
print(f'Predicted attack : {predicted_attack.name}')

q_vals = defender.q_values(dummy_state)
print(f'Q-values         : {dict(zip(HoneypotAction.names(), q_vals.round(3)))}')